# promote

Carries Saved tables from your Dev Database to your Prod Database.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Prod Database**: the database others read your finished Saved tables from, which a table reaches only once promoted from the Dev Database: `PROD` in admin/databases.py.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Reads itself**: a table whose own script reads it, such as a running total, seeded once from its original before its first write.
- **Original**: the table a converted script writes, in the warehouse already, which its Saved table copies; for a query alone, such as one Copilot wrote from a workbook, the rows the query gives.
- **Session setting**: a `SET` line sent before a Statement, which it needs to run.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Reference sheet**: references/reference_sheet.csv, one table listing every column of every table the Project has a Table reference for.
- **Database module**: a file such as references/ops.py, which gives each table of one database by name, such as ops.jobs.
- **Key**: the columns whose values name exactly one row of a table.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Hive**: the SQL your warehouse runs.
- **Statement**: a query your code builds, which becomes one Hive string.

## Before step 1

Fill in the names cell below and save the notebook: each call reads it, and so do `project.check()`, which names the next step, and `project.do_next()`, the cell before the last.

## Decision rules

- If you run the steps yourself, run each step's cell in turn, in order.
- If you want them run for you, run the last cell, `project.run_workflow(...)`, in place of their cells. It runs every step left in turn, up to the first only you can do, and says where it stopped: go on from that step.
- If you want only the steps that change files run for you, run `project.do_next()`, the cell before the last. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each dev table in `TABLES` has a prod copy: its Table folder in tables/ and its Table reference in references/.
- Each prod copy's Table reference has its `location=` and leaves no Decision open.
- admin/sent.csv records each prod copy created and written, and each that reads itself seeded.
- references/reference_sheet.csv lists each prod copy's columns.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = ["mart_dev.team_fails", "mart_dev.job_runs_counted", "mart_dev.job_fails_counted"]  # the dev tables, by their names in your Dev Database, such as ["mart_dev.daily_job_runs"]


In [ ]:
DECISIONS = {}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"location": "/data/team/mart_dev.db/weekly",
#                          "key": ["job_id"], "types": {"fails": "bigint"}}}
# "key": None says no columns name one row. project.check() names each one open;
# a Key proposed from a query is made, but an answer here changes it

## 1. Promote each table: a prod copy of its folder and Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** a prod copy of each dev table in `TABLES`, its own Table folder and Table reference, each reading the prod copies of the others promoted with it.
- **Action:** Run the cell below.
- **Done when** each dev table in `TABLES` has its prod copy's Table folder in tables/; then step 2.

In [ ]:
project.promote(TABLES)

## 2. Answer each prod table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each prod copy's open Decision: its location, the folder its files go in.
- **Action:** Write each answer in the decisions cell, `DECISIONS`, by the prod table's name as `project.check()` writes it, then save the notebook.
- **Done when** the decisions cell answers each Decision `project.check()` names for the prod copies; then step 3.

## 3. Write each answer into its prod table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its prod copy's Table reference, which then has its `location=`.
- **Action:** Run the cell below.
- **Done when** each prod copy's Table reference leaves no Decision open; then step 4.

In [ ]:
project.write_decisions(DECISIONS)

## 4. Create each table in prod

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy made in your Prod Database.
- **Action:** Run the cell below. It sends each prod copy's `create()` through your execute.
- **Done when** admin/sent.csv records a create of each prod copy; then step 5.

In [ ]:
for table in project.promoted_tables(TABLES):
    run_execute(project.load(table).create(), execute)

## 5. Seed each prod table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy that reads itself holding its original's rows, once: a prod copy's original is its dev copy's original.
- **Action:** Run the cell below. It sends the Session setting their Partitions need, then each such prod copy's `seed()`, through your execute. If no prod copy reads itself, this step is done once step 4 is.
- **Done when** admin/sent.csv records the seed of each prod copy that reads itself; then step 6.

In [ ]:
run_execute(hive_settings("dynamic_partitions"), execute)
for table in project.promoted_tables(TABLES, seeded=True):
    run_execute(project.load(table).seed(), execute)

## 6. Write each table in prod

It is a call to the warehouse, yours to run.

- **Goal:** each prod copy's rows written in your Prod Database.
- **Action:** Run the cell below. `project.run_tables` sends each prod copy's `write()` through your execute.
- **Done when** admin/sent.csv records each prod copy written; then step 7.

In [ ]:
project.run_tables(tables=project.promoted_tables(TABLES), execute=execute)

## 7. Refresh the Reference sheet and the Database modules

It changes only files, so project.do_next() runs it for you.

- **Goal:** each prod copy in the Reference sheet and in its Database module.
- **Action:** Run the cell below.
- **Done when** references/reference_sheet.csv is written again after each prod copy's Table reference last changed; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.refresh_reference_sheet(project.promoted_tables(TABLES))

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("promote", query=query, execute=execute)